In [ ]:
# BLOCK 1: IMPORT STUFF
# ============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os
from PIL import Image
import numpy as np

# check if GPU is working
print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found, using CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# some settings
DATA_DIR = "/kaggle/input/human-face-emotions/Data"
img_size = (128, 128)
batch_size = 32
num_workers = 4  # for loading data faster
epochs = 30
checkpoint_path = "/content/best_model.pth"

# set seed so results are reproducible
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("samithsachidanandan/human-face-emotions")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)

In [ ]:
# ============================================
# BLOCK 2: LOAD THE DATA
# ============================================
# define image transformations
data_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),  # this converts images to tensors and scales to [0,1]
])

# Correct the DATA_DIR to point to the downloaded dataset
DATA_DIR = path + '/Data'

# load all the images
full_data = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
class_names = full_data.classes
print("Found these emotion classes:", class_names)

# split into train and validation (80-20 split)
train_size = int(0.8 * len(full_data))
val_size = len(full_data) - train_size
torch.manual_seed(123)  # for consistent splits
train_data, val_data = random_split(full_data, [train_size, val_size])

print(f"Training images: {len(train_data)}")
print(f"Validation images: {len(val_data)}")

# create data loaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                         num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True)

In [ ]:
# ============================================
# BLOCK 3: BUILD THE CNN MODEL
# ============================================
class EmotionNet(nn.Module):
    def __init__(self, num_classes):
        super(EmotionNet, self).__init__()

        # first conv block
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.pool1 = nn.MaxPool2d(2, 2)

        # second conv block
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool2 = nn.MaxPool2d(2, 2)

        # third conv block
        self.conv3 = nn.Conv2d(64, 128, 3)
        self.pool3 = nn.MaxPool2d(2, 2)

        # fully connected layers
        # after all the pooling, we end up with 128 channels of 14x14 images
        self.fc1 = nn.Linear(128 * 14 * 14, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # conv layers with relu
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)

        x = torch.relu(self.conv2(x))
        x = self.pool2(x)

        x = torch.relu(self.conv3(x))
        x = self.pool3(x)

        # flatten
        x = x.view(x.size(0), -1)

        # dense layers
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # no softmax here, CrossEntropyLoss handles it

        return x

# create model and move to GPU
model = EmotionNet(len(class_names))
model = model.to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# ============================================
# BLOCK 4: EARLY STOPPING AND CHECKPOINTING
# ============================================
class EarlyStopping:
    """stops training if validation loss doesn't improve"""
    def __init__(self, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.should_stop = False
        self.best_weights = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_weights = model.state_dict().copy()
            print(f' Initial validation loss: {val_loss:.4f}')
        elif val_loss > self.best_loss:
            self.counter += 1
            print(f' EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.should_stop = True
                print('🛑 Early stopping triggered!')
        else:
            print(f'📉 Validation loss improved from {self.best_loss:.4f} to {val_loss:.4f}. Saving checkpoint.')
            self.best_loss = val_loss
            self.best_weights = model.state_dict().copy()
            self.counter = 0

    def restore_best_weights(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)
            print('✅ Restored best model weights')

class ModelCheckpoint:
    """saves model when val loss improves"""
    def __init__(self, filepath):
        self.filepath = filepath
        self.best_loss = float('inf')

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            torch.save({'model_state_dict': model.state_dict(),
                       'val_loss': val_loss}, self.filepath)

# initialize callbacks
checkpoint = ModelCheckpoint(checkpoint_path)
early_stopping = EarlyStopping(patience=5)

In [ ]:
# ============================================
# BLOCK 5: TRAINING LOOP
# ============================================
# track metrics
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print("\nStarting training...")
for epoch in range(epochs):
    print(f'\n{"="*60}\nEpoch {epoch+1}/{epochs}\n{"="*60}')

    # TRAINING
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # zero gradients
        optimizer.zero_grad()

        # forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # backward
        loss.backward()
        optimizer.step()

        # stats
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # VALIDATION
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_loss / total
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f'\nResults: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    # save checkpoint and check early stopping
    checkpoint(val_loss, model)
    early_stopping(val_loss, model)

    if early_stopping.should_stop:
        print('\n Stopping early!')
        early_stopping.restore_best_weights(model)
        break

# save final model
final_model_path = "/content/final_emotion_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'class_names': class_names,
    'train_losses': train_losses,
    'train_accs': train_accs,
    'val_losses': val_losses,
    'val_accs': val_accs
}, final_model_path)
print(f"\n Final model saved to {final_model_path}")


Starting training...

Epoch 1/30


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



Results: Train Loss: 1.3738 | Train Acc: 0.4231 | Val Loss: 1.1979 | Val Acc: 0.5045
 Initial validation loss: 1.1979

Epoch 2/30


In [ ]:
# ============================================
# BLOCK 6: EVALUATE THE MODEL
# ============================================
print("\n" + "="*60)
print("EVALUATION")
print("="*60)

# get predictions on validation set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# calculate final accuracy
val_accuracy = (all_preds == all_labels).mean()
print(f"Final Validation Accuracy: {val_accuracy:.4f}")

# classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

# plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_accs, label='Train Accuracy')
ax1.plot(val_accs, label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(train_losses, label='Train Loss')
ax2.plot(val_losses, label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# show some sample predictions
model.eval()
sample_images = []
sample_labels = []
sample_preds = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        sample_images = images[:5].cpu()
        sample_labels = labels[:5].cpu().numpy()
        sample_preds = probs[:5].cpu().numpy()
        break

# show images with predictions
plt.figure(figsize=(15, 5))
for i in range(len(sample_images)):
    plt.subplot(1, 5, i+1)
    img = sample_images[i].permute(1, 2, 0).numpy()
    img = (img * 255).astype(np.uint8)
    plt.imshow(img)
    plt.axis('off')
    pred_class = np.argmax(sample_preds[i])
    true_class = class_names[sample_labels[i]]
    pred_name = class_names[pred_class]
    plt.title(f"True: {true_class}\nPred: {pred_name}")
plt.tight_layout()
plt.show()

# FIXED: Show prediction probabilities - one figure with all samples
num_samples = len(sample_images)
fig, axes = plt.subplots(1, num_samples, figsize=(5*num_samples, 4))
if num_samples == 1:
    axes = [axes]  # make it iterable

for i in range(num_samples):
    axes[i].bar(class_names, sample_preds[i])
    axes[i].set_title(f"True: {class_names[sample_labels[i]]}")
    axes[i].set_ylabel("Probability")
    axes[i].set_xlabel("Emotion")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# BONUS: GRAD-CAM VISUALIZATION
# ============================================
def get_gradcam(img_tensor, model, target_layer):
    """generate grad-cam heatmap"""
    model.eval()

    gradients = []
    activations = []

    # FIXED: Use register_full_backward_hook instead of register_backward_hook
    def save_gradient(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    def save_activation(module, input, output):
        activations.append(output)

    # register hooks
    h1 = target_layer.register_forward_hook(save_activation)
    h2 = target_layer.register_full_backward_hook(save_gradient)  # FIXED

    # forward pass
    output = model(img_tensor)
    pred_class = output.argmax(dim=1).item()

    # backward pass
    model.zero_grad()
    output[0, pred_class].backward()

    # remove hooks
    h1.remove()
    h2.remove()

    # compute grad-cam
    grads = gradients[0].cpu().data.numpy()[0]
    acts = activations[0].cpu().data.numpy()[0]

    weights = np.mean(grads, axis=(1, 2))
    cam = np.zeros(acts.shape[1:], dtype=np.float32)

    for i, w in enumerate(weights):
        cam += w * acts[i]

    cam = np.maximum(cam, 0)
    cam = cam / (np.max(cam) + 1e-10)

    return cam, pred_class

# visualize grad-cam for first 3 images
model.eval()
gradcam_images = []
gradcam_heatmaps = []
gradcam_preds = []

for images, labels in val_loader:
    for idx in range(min(3, len(images))):  # Get 3 samples
        img = images[idx:idx+1].to(device)
        img.requires_grad = True

        heatmap, pred_idx = get_gradcam(img, model, model.conv3)

        # prepare image for display
        display_img = img[0].cpu().detach().permute(1, 2, 0).numpy()
        display_img = (display_img * 255).astype(np.uint8)

        # resize heatmap to match image size
        heatmap_resized = np.array(Image.fromarray(heatmap).resize(
            (display_img.shape[1], display_img.shape[0]),
            Image.BILINEAR
        ))

        gradcam_images.append(display_img)
        gradcam_heatmaps.append(heatmap_resized)
        gradcam_preds.append(pred_idx)

    break

# Show all Grad-CAMs in one figure
fig, axes = plt.subplots(1, len(gradcam_images), figsize=(6*len(gradcam_images), 6))
if len(gradcam_images) == 1:
    axes = [axes]

for idx in range(len(gradcam_images)):
    axes[idx].imshow(gradcam_images[idx])
    axes[idx].imshow(gradcam_heatmaps[idx], cmap='jet', alpha=0.4)
    axes[idx].set_title(f"Grad-CAM: {class_names[gradcam_preds[idx]]}")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
pip install gradio


In [ ]:
# ============================================
# BLOCK 7: INFERENCE + GRADIO UI
# ============================================
import gradio as gr

# ---------- 1) helper: load trained model ----------
def load_trained_model(
    checkpoint_file="/content/best_model.pth",
    fallback_file="/content/final_emotion_model.pth",
    device=device
):
    model = EmotionNet(len(class_names)).to(device)

    ckpt_path = None
    if os.path.exists(checkpoint_file):
        ckpt_path = checkpoint_file
    elif os.path.exists(fallback_file):
        ckpt_path = fallback_file

    if ckpt_path is not None:
        print(f"Loading weights from: {ckpt_path}")
        checkpoint_data = torch.load(ckpt_path, map_location=device)
        state_dict = checkpoint_data.get("model_state_dict", checkpoint_data)
        model.load_state_dict(state_dict)
    else:
        print("⚠️ No checkpoint file found, using current in-memory model.")

    model.eval()
    return model

inference_model = load_trained_model()

# ---------- 2) Grad-CAM function reused (slightly adapted) ----------
def gradcam_for_image(img_tensor, model, target_layer):
    model.eval()
    gradients = []
    activations = []

    def save_activation(module, input, output):
        activations.append(output)

    def save_gradient(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    h1 = target_layer.register_forward_hook(save_activation)
    h2 = target_layer.register_full_backward_hook(save_gradient)

    output = model(img_tensor)
    pred_class = output.argmax(dim=1).item()

    model.zero_grad()
    output[0, pred_class].backward()

    h1.remove()
    h2.remove()

    grads = gradients[0].detach().cpu().numpy()[0]
    acts = activations[0].detach().cpu().numpy()[0]

    weights = np.mean(grads, axis=(1, 2))
    cam = np.zeros(acts.shape[1:], dtype=np.float32)

    for i, w in enumerate(weights):
        cam += w * acts[i]

    cam = np.maximum(cam, 0)
    cam = cam / (np.max(cam) + 1e-10)

    return cam, pred_class

# ---------- 3) prediction function used by Gradio ----------
def predict_emotion(image: Image.Image):
    """
    image: PIL image from gradio
    returns: (predicted_class_name, bar_plot, gradcam_overlay_image)
    """

    # convert PIL -> tensor
    transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor()
    ])

    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = inference_model(img_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
        pred_idx = int(np.argmax(probs))
        pred_name = class_names[pred_idx]

    # -------- create bar plot of probabilities --------
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(class_names, probs)
    ax.set_ylabel("Probability")
    ax.set_xlabel("Emotion")
    ax.set_title(f"Prediction Probabilities (Pred: {pred_name})")
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

    # -------- create Grad-CAM overlay --------
    cam, _ = gradcam_for_image(img_tensor, inference_model, inference_model.conv3)

    # original image (resized) as numpy
    disp_img = transform(image).permute(1, 2, 0).numpy()
    disp_img = (disp_img * 255).astype(np.uint8)

    # resize cam to image size
    cam_resized = np.array(Image.fromarray(cam).resize(
        (disp_img.shape[1], disp_img.shape[0]),
        Image.BILINEAR
    ))

    # normalize cam_resized to [0, 255] and apply colormap
    cam_resized_uint8 = (cam_resized * 255).astype(np.uint8)
    heatmap = plt.cm.jet(cam_resized_uint8)[:, :, :3]  # remove alpha channel
    heatmap = (heatmap * 255).astype(np.uint8)

    overlay = (0.4 * heatmap + 0.6 * disp_img).astype(np.uint8)

    # convert overlay back to PIL for Gradio
    overlay_pil = Image.fromarray(overlay)

    return pred_name, fig, overlay_pil

# ---------- 4) build Gradio Interface ----------
demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(type="pil", label="Upload a face image"),
    outputs=[
        gr.Label(label="Predicted Emotion"),
        gr.Plot(label="Class Probabilities"),
        gr.Image(label="Grad-CAM Visualization")
    ],
    title="Human Face Emotions Classifier",
    description="Upload a face image and the model will predict the emotion and show Grad-CAM."
)

# launch the app (in notebook it will show inline link/UI)
demo.launch(share=False)


In [ ]:
# ============================================
# BLOCK 7: SIMPLE GRADIO UI (WEB & CAMERA)
# ============================================
import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import os

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define class_names to match the trained model (5 classes)
class_names = ['Angry', 'Fear', 'Happy', 'Sad', 'Suprise']

# Model Definition (copied from BLOCK 3 to make this cell self-contained)
class EmotionNet(nn.Module):
    def __init__(self, num_classes):
        super(EmotionNet, self).__init__()

        # first conv block
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.pool1 = nn.MaxPool2d(2, 2)

        # second conv block
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool2 = nn.MaxPool2d(2, 2)

        # third conv block
        self.conv3 = nn.Conv2d(64, 128, 3)
        self.pool3 = nn.MaxPool2d(2, 2)

        # fully connected layers
        self.fc1 = nn.Linear(128 * 14 * 14, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # conv layers with relu
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)

        x = torch.relu(self.conv2(x))
        x = self.pool2(x)

        x = torch.relu(self.conv3(x))
        x = self.pool3(x)

        # flatten
        x = x.view(x.size(0), -1)

        # dense layers
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # no softmax here, CrossEntropyLoss handles it

        return x

# Load the trained model (similar to qnxX5ZD8Ss0E)
model = EmotionNet(len(class_names)).to(device)
checkpoint_file = "/content/best_model.pth"
fallback_file = "/content/final_emotion_model.pth"

ckpt_path = None
if os.path.exists(checkpoint_file):
    ckpt_path = checkpoint_file
elif os.path.exists(fallback_file):
    ckpt_path = fallback_file

if ckpt_path is not None:
    print(f"Loading weights from: {ckpt_path}")
    checkpoint_data = torch.load(ckpt_path, map_location=device)
    state_dict = checkpoint_data.get("model_state_dict", checkpoint_data)
    model.load_state_dict(state_dict)
else:
    print("⚠️ No checkpoint file found, model weights not loaded.")

model.eval()

# Data transformations
img_size = (128, 128) # Ensure img_size is defined if not global
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor()
])

# Grad-CAM function (adapted for clarity and robustness)
def simple_gradcam(img_tensor, model, target_layer):
    model.eval()
    gradients = []
    activations = []

    def f_hook(module, inp, out):
        activations.append(out)

    def b_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    h1 = target_layer.register_forward_hook(f_hook)
    h2 = target_layer.register_full_backward_hook(b_hook)

    output = model(img_tensor)
    pred_class = output.argmax(dim=1).item()

    model.zero_grad()
    # Ensure only the specific output for the predicted class contributes to gradients
    output[0, pred_class].backward()

    h1.remove()
    h2.remove()

    acts = activations[0].detach().cpu().numpy()[0]
    grads = gradients[0].detach().cpu().numpy()[0]

    weights = np.mean(grads, axis=(1, 2))
    cam = np.zeros(acts.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * acts[i]

    cam = np.maximum(cam, 0)
    cam = cam / (cam.max() + 1e-10)
    return cam, pred_class

def predict_emotion(img):
    if img is None:
        return "No image", {}, None

    # numpy → PIL
    pil = Image.fromarray(img).convert("RGB")

    # preprocess
    x = transform(pil).unsqueeze(0).to(device)

    # prediction
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    pred_idx = int(np.argmax(probs))
    pred_label = class_names[pred_idx]

    # Grad-CAM
    cam, _ = simple_gradcam(x, model, model.conv3)

    # resize heatmap to image size
    cam_img = (cam * 255).astype(np.uint8)
    cam_img = Image.fromarray(cam_img).resize(pil.size)
    cam_arr = np.array(cam_img)

    # apply colormap
    heatmap = plt.cm.jet(cam_arr / 255.0)[..., :3]   # (H,W,3) in [0,1]
    heatmap = (heatmap * 255).astype(np.uint8)

    # overlay with original image
    base = np.array(pil).astype(np.float32)
    overlay = (0.6 * base + 0.4 * heatmap).astype(np.uint8)

    # probabilities as dict
    prob_dict = {cls: float(p) for cls, p in zip(class_names, probs)}

    return pred_label, prob_dict, overlay

demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(
        sources=["webcam", "upload"],   # FIXED: Changed 'source' to 'sources' and added 'upload'
        type="numpy",
        label="Image / Webcam"
    ),
    outputs=[
        gr.Label(label="Predicted Emotion"),
        gr.Label(label="Probabilities"),
        gr.Image(label="Grad-CAM")
    ],
    title="Human Face Emotion Recognition",
    css="""
        body {
            background: #0b3d91 !important;
        }
        .gradio-container {
            background: #0b3d91 !important;
        }
    """
)

demo.launch()